In [2]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization (for sanity checks)
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)

print("Setup complete ✅")

Setup complete ✅


In [3]:
# Load dataset
df = pd.read_excel("GEFCom2014-E.xlsx")

# Preview
df.head()

,Date,Hour,load,T
0,2004-01-01,1,NaN,37.333333
1,2004-01-01,2,NaN,37.666667
2,2004-01-01,3,NaN,37.000000
3,2004-01-01,4,NaN,36.333333
4,2004-01-01,5,NaN,36.000000


In [4]:
# Rename columns (clean naming)
df.columns = ['date', 'hour', 'load', 'temp']

# Convert date
df['date'] = pd.to_datetime(df['date'])

# Sort properly
df = df.sort_values(by=['date', 'hour']).reset_index(drop=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96432 entries, 0 to 96431
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    96432 non-null  datetime64[us]
 1   hour    96432 non-null  int64         
 2   load    78888 non-null  float64       
 3   temp    96432 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1)
memory usage: 2.9 MB


In [5]:
# Check missing values
print(df.isnull().sum())

# Fill load using interpolation (important)
df['load'] = df['load'].interpolate(method='linear')

# Fill temp if needed
df['temp'] = df['temp'].interpolate(method='linear')

print("Missing values handled ✅")

date        0
hour        0
load    17544
temp        0
dtype: int64
Missing values handled ✅


In [6]:
# Extract time features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# Weekend flag
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

df.head()

,date,hour,load,temp,day_of_week,month,day,is_weekend
0,2004-01-01,1,NaN,37.333333,3,1,1,0
1,2004-01-01,2,NaN,37.666667,3,1,1,0
2,2004-01-01,3,NaN,37.000000,3,1,1,0
3,2004-01-01,4,NaN,36.333333,3,1,1,0
4,2004-01-01,5,NaN,36.000000,3,1,1,0


In [7]:
# Lag features
df['lag_1'] = df['load'].shift(1)
df['lag_24'] = df['load'].shift(24)
df['lag_168'] = df['load'].shift(168)

# Drop initial NaNs caused by lag
df = df.dropna().reset_index(drop=True)

print("Lag features created ✅")

Lag features created ✅


In [8]:
h = df['hour']

# Improved temperature variation
df['temp_z1'] = df['temp'] + 2*np.sin((h-14)/24 * 2*np.pi) + 1.5   # warmer region
df['temp_z2'] = df['temp'] - 2*np.sin((h-10)/24 * 2*np.pi) - 1.5   # cooler region
df['temp_z3'] = df['temp'] + np.random.normal(0, 1.2, len(df))     # noisy region

print("Zone temperatures created ✅")

Zone temperatures created ✅


In [9]:
# Normalize temperature
temp_norm_z1 = (df['temp_z1'] - df['temp_z1'].min()) / (df['temp_z1'].max() - df['temp_z1'].min())
temp_norm_z2 = (df['temp_z2'] - df['temp_z2'].min()) / (df['temp_z2'].max() - df['temp_z2'].min())

# Base patterns (WITHOUT multiplying by load yet)
res_pattern = 0.4 + 0.1*np.sin((df['hour']-7)/24 * 2*np.pi) + 0.04*temp_norm_z1
com_pattern = 0.35 + 0.1*np.sin((df['hour']-13)/24 * 2*np.pi) + 0.03*temp_norm_z2

# 🔴 IMPORTANT: Normalize patterns so they don’t exceed 1
total_pattern = res_pattern + com_pattern

res_ratio = res_pattern / (total_pattern + 1e-6)
com_ratio = com_pattern / (total_pattern + 1e-6)

# Scale safely (leave room for industrial)
res_ratio *= 0.75
com_ratio *= 0.75

# Create zones
df['zone_residential'] = df['load'] * res_ratio
df['zone_commercial'] = df['load'] * com_ratio

# Industrial = residual (guaranteed non-negative)
df['zone_industrial'] = df['load'] - df['zone_residential'] - df['zone_commercial']

print("Pseudo hierarchy created ✅")

Pseudo hierarchy created ✅


In [10]:
df['sum_zones'] = (
    df['zone_residential'] + 
    df['zone_commercial'] + 
    df['zone_industrial']
)

error = np.abs(df['sum_zones'] - df['load']).mean()
print("Reconstruction error:", error)

Reconstruction error: 1.9279970573978328e-14


In [11]:
df['zone_residential'] *= np.random.normal(1, 0.02, len(df))
df['zone_commercial'] *= np.random.normal(1, 0.02, len(df))

# Recompute industrial to maintain consistency
df['zone_industrial'] = df['load'] - df['zone_residential'] - df['zone_commercial']

print("Noise added ✅")

Noise added ✅


In [12]:
df.head()
df.describe()

,date,hour,load,temp,day_of_week,month,day,is_weekend,lag_1,lag_24,lag_168,temp_z1,temp_z2,temp_z3,zone_residential,zone_commercial,zone_industrial,sum_zones
count,78720,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000
mean,2010-07-05 12:00:00,12.500000,3307.522764,47.623209,2.999085,6.535061,15.753659,0.285671,3307.520058,3307.441502,3308.013758,49.123209,46.123209,47.623722,1331.292512,1149.449295,826.780957,3307.522764
min,2006-01-08 00:00:00,1.000000,1811.000000,-18.000000,0.000000,1.000000,1.000000,0.000000,1811.000000,1811.000000,1811.000000,-18.431852,-18.982362,-19.605349,693.735985,591.841260,447.913303,1811.000000
25%,2008-04-06 18:00:00,6.750000,2845.000000,32.666667,1.000000,4.000000,8.000000,0.000000,2845.000000,2845.000000,2845.000000,34.232051,31.500000,32.828280,1080.307630,958.348496,709.404657,2845.000000
50%,2010-07-05 12:00:00,12.500000,3381.000000,48.666667,3.000000,7.000000,16.000000,0.000000,3381.000000,3381.000000,3382.000000,49.982362,47.166667,48.600095,1329.495259,1109.840317,839.425749,3381.000000
75%,2012-10-02 06:00:00,18.250000,3708.000000,63.000000,5.000000,10.000000,23.000000,1.000000,3708.000000,3708.000000,3709.000000,64.565384,61.752453,63.124912,1559.208604,1326.024831,929.694983,3708.000000
max,2014-12-31 00:00:00,24.000000,5506.000000,97.000000,6.000000,12.000000,31.000000,1.000000,5506.000000,5506.000000,5506.000000,99.914214,93.568148,96.460462,2467.931988,2130.101126,1456.773056,5506.000000
std,NaN,6.922231,579.824033,19.190627,2.000546,3.442846,8.791906,0.451736,579.824355,579.793714,579.862178,19.439976,18.905500,19.225360,294.733788,252.865945,149.248985,579.824033


In [13]:
df.head()

,date,hour,load,temp,day_of_week,month,day,is_weekend,lag_1,lag_24,lag_168,temp_z1,temp_z2,temp_z3,zone_residential,zone_commercial,zone_industrial,sum_zones
0,2006-01-08,1,2915.0,15.000000,6,1,8,1,3132.0,2895.0,3010.0,17.017638,14.914214,15.596057,1035.592930,1201.566544,677.840526,2915.0
1,2006-01-08,2,2803.0,15.333333,6,1,8,1,2915.0,2799.0,2853.0,16.833333,15.565384,15.167416,1036.329952,1093.417381,673.252667,2803.0
2,2006-01-08,3,2742.0,16.000000,6,1,8,1,2803.0,2747.0,2758.0,16.982362,16.431852,16.777226,1086.702380,1003.517892,651.779729,2742.0
3,2006-01-08,4,2730.0,15.666667,6,1,8,1,2742.0,2745.0,2705.0,16.166667,16.166667,17.494302,1133.451039,942.319791,654.229169,2730.0
4,2006-01-08,5,2749.0,16.333333,6,1,8,1,2730.0,2790.0,2709.0,16.419120,16.765185,16.052349,1168.744653,889.584119,690.671228,2749.0


In [14]:
# Temperature lag features
df['temp_lag_1'] = df['temp'].shift(1)
df['temp_lag_24'] = df['temp'].shift(24)

print("Temperature lag features created ✅")

Temperature lag features created ✅


In [15]:
# Rolling temperature (use past data only)
df['temp_roll_mean_24'] = df['temp'].shift(1).rolling(24).mean()
df['temp_roll_std_24'] = df['temp'].shift(1).rolling(24).std()

print("Rolling temperature features created ✅")

Rolling temperature features created ✅


In [16]:
df.describe()

,date,hour,load,temp,day_of_week,month,day,is_weekend,lag_1,lag_24,...,temp_z2,temp_z3,zone_residential,zone_commercial,zone_industrial,sum_zones,temp_lag_1,temp_lag_24,temp_roll_mean_24,temp_roll_std_24
count,78720,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,...,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78720.000000,78719.000000,78696.000000,78696.000000,78696.000000
mean,2010-07-05 12:00:00,12.500000,3307.522764,47.623209,2.999085,6.535061,15.753659,0.285671,3307.520058,3307.441502,...,46.123209,47.623722,1331.292512,1149.449295,826.780957,3307.522764,47.623619,47.632434,47.631847,5.538399
min,2006-01-08 00:00:00,1.000000,1811.000000,-18.000000,0.000000,1.000000,1.000000,0.000000,1811.000000,1811.000000,...,-18.982362,-19.605349,693.735985,591.841260,447.913303,1811.000000,-18.000000,-18.000000,-4.722222,0.386726
25%,2008-04-06 18:00:00,6.750000,2845.000000,32.666667,1.000000,4.000000,8.000000,0.000000,2845.000000,2845.000000,...,31.500000,32.828280,1080.307630,958.348496,709.404657,2845.000000,32.666667,33.000000,33.440972,3.577029
50%,2010-07-05 12:00:00,12.500000,3381.000000,48.666667,3.000000,7.000000,16.000000,0.000000,3381.000000,3381.000000,...,47.166667,48.600095,1329.495259,1109.840317,839.425749,3381.000000,48.666667,48.666667,48.986111,5.350328
75%,2012-10-02 06:00:00,18.250000,3708.000000,63.000000,5.000000,10.000000,23.000000,1.000000,3708.000000,3708.000000,...,61.752453,63.124912,1559.208604,1326.024831,929.694983,3708.000000,63.000000,63.000000,63.694444,7.290291
max,2014-12-31 00:00:00,24.000000,5506.000000,97.000000,6.000000,12.000000,31.000000,1.000000,5506.000000,5506.000000,...,93.568148,96.460462,2467.931988,2130.101126,1456.773056,5506.000000,97.000000,97.000000,87.888889,16.138266
std,NaN,6.922231,579.824033,19.190627,2.000546,3.442846,8.791906,0.451736,579.824355,579.793714,...,18.905500,19.225360,294.733788,252.865945,149.248985,579.824033,19.190404,19.186085,18.238666,2.522301


In [17]:
base_temp = 65

df['CDD'] = (df['temp'] - base_temp).clip(lower=0)
df['HDD'] = (base_temp - df['temp']).clip(lower=0)

In [18]:
# Rolling load statistics (past only)
df['load_roll_mean_24'] = df['load'].shift(1).rolling(24).mean()
df['load_roll_std_24'] = df['load'].shift(1).rolling(24).std()

print("Load rolling features created ✅")

Load rolling features created ✅


In [19]:
df = df.dropna().reset_index(drop=True)

print("NaNs removed after feature creation ✅")

NaNs removed after feature creation ✅


In [20]:
df[['temp','temp_lag_1','temp_roll_mean_24','CDD','HDD']].head(10)

,temp,temp_lag_1,temp_roll_mean_24,CDD,HDD
0,20.666667,20.000000,21.222222,0.0,44.333333
1,22.000000,20.666667,21.458333,0.0,43.000000
2,23.333333,22.000000,21.736111,0.0,41.666667
3,23.333333,23.333333,22.041667,0.0,41.666667
4,24.333333,23.333333,22.361111,0.0,40.666667
5,24.333333,24.333333,22.694444,0.0,40.666667
6,25.000000,24.333333,22.944444,0.0,40.000000
7,25.666667,25.000000,23.208333,0.0,39.333333
8,25.666667,25.666667,23.541667,0.0,39.333333
9,26.000000,25.666667,23.819444,0.0,39.000000


In [ ]:
df.describe()